# Shared archival pipeline — Colab model comparisons
Synthetic data only. This notebook imports the actual server text/vision prompts from a versioned GitHub checkout. It does not reproduce Box ingestion, original-file extraction or server services inside Colab.

Choose Runtime → Change runtime type → GPU. Free GPU availability is not guaranteed. No paid APIs are used. Run one model at a time. This small integration smoke set is not the complete validation program or proof of archival accuracy.

The first cell resolves main once, then checks out that exact commit. For a repeat run, put the recorded commit into CODE_REF. Never upload original accession files or credentials to this notebook.


In [ ]:
from pathlib import Path
import subprocess, os, sys
REPO_URL = "https://github.com/terbe2022/RIMS-Archival-Project-.git"
CODE_REF = "main"  # replace with the recorded commit for reproducibility
CHECKOUT = Path("/content/archival-pipeline")
if CHECKOUT.exists():
    raise RuntimeError("Checkout already exists. Restart the runtime or choose a fresh checkout path; do not overwrite work.")
subprocess.run(["git", "clone", REPO_URL, str(CHECKOUT)], check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=CHECKOUT, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip()
print("Pipeline code commit:", COMMIT)
os.chdir(CHECKOUT)


In [ ]:
%pip install "transformers>=4.51,<5" "accelerate>=1,<2" "pyarrow>=14" "Pillow>=10" requests sentencepiece


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running models.")
sys.path.insert(0, str(CHECKOUT))
from validation.pipeline_model_compare import run


## First: offline contract smoke check
No model calls. Stub results check the harness only and must not be reported as model performance.


In [ ]:
stub = run(["stub"], backend="stub", out="local-runs/contract-smoke.json")
assert all(row["metrics"]["schema_valid"] for row in stub["rows"])
print("Offline contract checks passed")


## Text comparison
Begin with one small Qwen model. Add another model ID only after the first smoke run works. All candidates receive the same pipeline prompt and synthetic cases. Preserve failures. Model download licences and runtime memory limits still apply.


In [ ]:
TEXT_MODELS = ["Qwen/Qwen2.5-0.5B-Instruct"]
# Optional second candidate: "Qwen/Qwen2.5-1.5B-Instruct"
text_report = run(TEXT_MODELS, lane="text", backend="hf", out="local-runs/text-comparison.json")
for row in text_report["rows"]:
    print(row["model"], row["case"], row["metrics"], row.get("error", ""))


## Optional vision comparison
Uses two generated drawings, not accession images. This checks image-model integration and output schema; evaluate description quality separately with an agreed richer synthetic fixture set. The current server vision prompt is used unchanged.


In [ ]:
VISION_MODELS = ["HuggingFaceTB/SmolVLM-256M-Instruct"]
vision_report = run(VISION_MODELS, lane="vision", backend="hf", out="local-runs/vision-comparison.json")
for row in vision_report["rows"]:
    print(row["model"], row["case"], row["metrics"], row.get("error", ""))


## Download results
Review synthetic outputs before sharing. Record conclusions in a Gauri experiment report and submit reviewed non-sensitive results through a pull request. Do not commit notebook outputs, model weights or credentials. Raw run artifacts stay under ignored local-runs/.


In [ ]:
from google.colab import files
for name in ["contract-smoke.json", "text-comparison.json", "vision-comparison.json"]:
    path = CHECKOUT / "local-runs" / name
    if path.exists():
        files.download(str(path))
